In [1]:
!pip install lizard pydriller tqdm pandas requests scikit-learn

In [2]:
import os
import re
import requests
import pandas as pd

from collections import defaultdict, Counter
from itertools import islice
from tqdm import tqdm
from pydriller import Repository
import lizard

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split


In [3]:
TRAIN_DATA_FILE = "django_osv_dataset_5000.csv"

NEW_REPO_URL = "https://github.com/pallets/flask"
NEW_PROJECT_NAME = "Flask"
NEW_ECOSYSTEM = "PyPI"

NEW_DATA_RAW_FILE = "flask_raw_candidates.csv"
RESULT_FILE = "flask_risk_prediction_results.csv"
TOP50_FILE = "flask_top50_riskiest_files.csv"

FEATURE_COLS = [
    "loc", "func_cnt", "ccn", "avg_ccn", "max_ccn", "token_count",
    "avg_params", "file_commits", "unique_authors",
    "file_age_days", "security_touch_ratio"
]

SAFE_KEYWORDS = {
    "refactor", "cleanup", "clean up", "typo", "docs", "documentation",
    "style", "format", "test", "tests", "ci", "build", "bump", "version",
    "rename", "comment", "lint", "readme", "translation", "chore"
}


In [4]:
if not os.path.exists(TRAIN_DATA_FILE):
    raise FileNotFoundError(
        f"Не найден файл {TRAIN_DATA_FILE}. "
        "Сначала запусти предыдущий ноутбук django_osv_dataset_5000.ipynb"
    )

train_df = pd.read_csv(TRAIN_DATA_FILE)
print(train_df.shape)
train_df.head()


(5000, 17)


,file,hash,date,author,message,is_vuln,file_commits,unique_authors,file_age_days,security_touch_ratio,loc,func_cnt,ccn,avg_ccn,max_ccn,token_count,avg_params
0,django\conf\urls\static.py,610876661ecdfbc1a3ca3c5458e1801313501c77,2017-02-22,Anton Samarchyan,Fixed typos in code comments.,0,9,6,2200,0.0000,13,1,4,4.000,4,113,3.000
1,tests\schema\tests.py,34b411762b50883d768d7b67e0a158ec39da8b09,2023-10-30,Tom Carrick,Fixed #34932 -- Restored varchar_pattern_ops/t...,0,250,75,3845,0.0000,4290,241,432,1.793,13,19233,1.187
2,django\contrib\formtools\tests\wizard\wizardte...,c4c2c99669ef9d58306856c47058b121bbecfe5f,2014-05-30,Florian Apolloner,Fixed #19905 -- Fixed leakage of file descript...,0,16,10,1021,0.0000,349,26,28,1.077,2,1753,1.231
3,tests\urlpatterns_reverse\tests.py,c1874176114cd94efcc58e3c449f4e2a116881a1,2024-03-04,Adam Johnson,Refs #9847 -- Added tests for handler403 resol...,0,124,50,4023,0.0484,1507,102,140,1.373,5,5933,1.000
4,tests\gis_tests\distapp\tests.py,aaacaeb0963c406c4fe6f68c6ae51f4a65878250,2015-06-24,Tim Graham,Renamed RemovedInDjangoXYWarnings for new road...,0,9,3,133,0.0000,370,24,84,3.500,10,3385,1.042


In [5]:
X = train_df[FEATURE_COLS].copy()
y = train_df["is_vuln"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

test_probs = model.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= 0.5).astype(int)

print("ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4))
print(classification_report(y_test, test_pred, digits=4))


ROC-AUC: 0.793
              precision    recall  f1-score   support

           0     0.9434    0.9444    0.9439       900
           1     0.4949    0.4900    0.4925       100

    accuracy                         0.8990      1000
   macro avg     0.7192    0.7172    0.7182      1000
weighted avg     0.8986    0.8990    0.8988      1000



In [6]:
feature_importance = (
    pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance": model.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance


,feature,importance
0,security_touch_ratio,0.364637
1,file_age_days,0.100584
2,token_count,0.074980
3,file_commits,0.067081
4,loc,0.066518
5,ccn,0.061250
6,avg_ccn,0.060020
7,avg_params,0.058425
8,unique_authors,0.052234
9,func_cnt,0.050004


In [7]:
def load_osv_records(package_name, ecosystem):
    payload = {
        "queries": [
            {
                "package": {
                    "name": package_name,
                    "ecosystem": ecosystem
                }
            }
        ]
    }

    response = requests.post(
        "https://api.osv.dev/v1/querybatch",
        json=payload,
        timeout=60
    )
    response.raise_for_status()
    batch_data = response.json()

    vuln_ids = []
    for item in batch_data.get("results", []):
        for vuln in item.get("vulns", []):
            if "id" in vuln:
                vuln_ids.append(vuln["id"])

    records = []
    for vuln_id in tqdm(vuln_ids, desc=f"Loading OSV advisories for {package_name}"):
        r = requests.get(f"https://api.osv.dev/v1/vulns/{vuln_id}", timeout=60)
        if r.status_code == 200:
            records.append(r.json())

    return records


def parse_osv_data(records):
    vuln_hashes = set()
    vuln_keywords = set()
    vuln_titles = {}

    for record in records:
        vuln_id = record.get("id", "")
        text_parts = []

        summary = record.get("summary", "") or ""
        details = record.get("details", "") or ""
        text_parts.extend([summary, details])

        if summary:
            vuln_titles[vuln_id] = summary

        for ref in record.get("references", []):
            url = ref.get("url", "")
            text_parts.append(url)

            commit_match = re.search(r"/commit/([0-9a-f]{7,40})", url)
            if commit_match:
                vuln_hashes.add(commit_match.group(1))

        for affected in record.get("affected", []):
            for rng in affected.get("ranges", []):
                if rng.get("type") == "GIT":
                    for event in rng.get("events", []):
                        fixed_hash = event.get("fixed")
                        if fixed_hash:
                            vuln_hashes.add(fixed_hash)

        full_text = " ".join(text_parts).lower()
        for word in re.findall(r"[a-zA-Z][a-zA-Z\-]{3,}", full_text):
            if word in {
                "security", "vulnerability", "injection", "xss", "csrf",
                "sqli", "bypass", "exposure", "disclosure", "traversal",
                "rce", "overflow", "deserialization", "spoofing",
                "privilege", "escalation", "authentication", "authorization",
                "redirect", "dos", "denial", "forgery", "cve"
            }:
                vuln_keywords.add(word)

    return vuln_hashes, vuln_keywords, vuln_titles


In [8]:
new_osv_records = load_osv_records(NEW_PROJECT_NAME, NEW_ECOSYSTEM)
NEW_VULN_HASHES, NEW_VULN_KEYWORDS, NEW_OSV_TITLES = parse_osv_data(new_osv_records)

print("OSV advisories:", len(new_osv_records))
print("Known fix hashes:", len(NEW_VULN_HASHES))
print("Keywords:", sorted(NEW_VULN_KEYWORDS))


Loading OSV advisories for Flask: 100%|██████████████████████████████████████████████████| 8/8 [00:02<00:00,  3.88it/s]

OSV advisories: 8
Known fix hashes: 5
Keywords: ['denial', 'disclosure', 'security', 'vulnerability']


In [9]:
def classify_commit_new_project(commit_hash, message):
    short_hash = (commit_hash or "")[:40]

    if short_hash in NEW_VULN_HASHES:
        return "vuln"

    if not message or len(message.strip()) < 4:
        return "skip"

    msg = message.lower()

    has_vuln_keyword = any(word in msg for word in NEW_VULN_KEYWORDS) or ("security" in msg) or ("cve-" in msg)
    has_safe_keyword = any(word in msg for word in SAFE_KEYWORDS)

    if has_vuln_keyword:
        return "vuln"

    if has_safe_keyword:
        return "safe"

    return "safe"


In [10]:
class TransferDatasetBuilder:
    def __init__(self, repo_url):
        self.repo_url = repo_url
        self.rows = []
        self.file_history = defaultdict(lambda: {
            "commits": 0,
            "authors": set(),
            "security_commits": 0,
            "first_seen": None
        })
        self.counters = Counter()

    def calc_code_metrics(self, code, filename):
        if not code or not str(code).strip():
            return None

        try:
            analysis = lizard.analyze_file.analyze_source_code(filename, code)
        except Exception:
            return None

        if not analysis or analysis.nloc == 0:
            return None

        funcs = analysis.function_list
        avg_ccn = sum(f.cyclomatic_complexity for f in funcs) / len(funcs) if funcs else 0
        max_ccn = max((f.cyclomatic_complexity for f in funcs), default=0)
        avg_params = sum(f.parameter_count for f in funcs) / len(funcs) if funcs else 0

        return {
            "loc": analysis.nloc,
            "func_cnt": len(funcs),
            "ccn": analysis.CCN,
            "avg_ccn": round(avg_ccn, 3),
            "max_ccn": max_ccn,
            "token_count": len(re.findall(r"\w+", code)),
            "avg_params": round(avg_params, 3)
        }

    def build_row(self, commit, modified_file, label):
        path = modified_file.new_path or modified_file.old_path
        if not path:
            return None

        ext = "." + path.split(".")[-1].lower() if "." in path else ""
        if ext not in {".py", ".js", ".ts", ".java", ".scala"}:
            return None

        code = modified_file.source_code_before if label == "vuln" else (modified_file.source_code or modified_file.source_code_before)
        metrics = self.calc_code_metrics(code, path)
        if not metrics:
            return None

        hist = self.file_history[path]
        file_age_days = 0
        if hist["first_seen"] is not None:
            file_age_days = max((commit.committer_date - hist["first_seen"]).days, 0)

        row = {
            "file": path,
            "hash": commit.hash,
            "date": commit.committer_date.date().isoformat(),
            "author": commit.author.name,
            "message": commit.msg[:150],
            "is_vuln": 1 if label == "vuln" else 0,
            "file_commits": hist["commits"],
            "unique_authors": len(hist["authors"]),
            "file_age_days": file_age_days,
            "security_touch_ratio": round(
                hist["security_commits"] / hist["commits"], 4
            ) if hist["commits"] else 0.0
        }
        row.update(metrics)
        return row

    def update_history(self, commit, modified_file, label):
        path = modified_file.new_path or modified_file.old_path
        if not path:
            return

        hist = self.file_history[path]
        hist["commits"] += 1
        hist["authors"].add(commit.author.email or commit.author.name)

        if hist["first_seen"] is None:
            hist["first_seen"] = commit.committer_date

        if label == "vuln":
            hist["security_commits"] += 1

    def extract(self, max_commits=25000):
        repo = Repository(self.repo_url, only_no_merge=True)
        for commit in tqdm(islice(repo.traverse_commits(), max_commits), total=max_commits, desc="Mining new project"):
            label = classify_commit_new_project(commit.hash, commit.msg)
            if label == "skip":
                self.counters["skipped_commits"] += 1
                continue

            for modified_file in commit.modified_files:
                row = self.build_row(commit, modified_file, label)
                if row is not None:
                    self.rows.append(row)
                    self.counters["rows"] += 1
                    self.counters[f"{label}_rows"] += 1

                self.update_history(commit, modified_file, label)

        return pd.DataFrame(self.rows)


In [11]:
builder = TransferDatasetBuilder(NEW_REPO_URL)
new_df = builder.extract(max_commits=25000)

new_df.to_csv(NEW_DATA_RAW_FILE, index=False)

print(new_df.shape)
new_df.head()


Mining new project:  15%|████████▋                                                | 3805/25000 [03:51<21:28, 16.45it/s]


(4241, 17)


,file,hash,date,author,message,is_vuln,file_commits,unique_authors,file_age_days,security_touch_ratio,loc,func_cnt,ccn,avg_ccn,max_ccn,token_count,avg_params
0,examples\apishowcase\apishowcase.py,33850c0ebd23ae615e6823993d441f46d80b1ff0,2010-04-06,Armin Ronacher,Initial checkin of stuff that exists so far.,0,0,0,0,0.0,16,3,4,1.333,2,114,0.333
1,examples\minitwit\minitwit.py,33850c0ebd23ae615e6823993d441f46d80b1ff0,2010-04-06,Armin Ronacher,Initial checkin of stuff that exists so far.,0,0,0,0,0.0,172,17,44,2.588,9,878,0.588
2,flask.py,33850c0ebd23ae615e6823993d441f46d80b1ff0,2010-04-06,Armin Ronacher,Initial checkin of stuff that exists so far.,0,0,0,0,0.0,185,28,49,1.750,6,1432,1.857
3,setup.py,b15ad394279fc3b7f998fa56857f334a7c0156f6,2010-04-06,Armin Ronacher,Added setup.py and README,0,0,0,0,0.0,17,0,0,0.000,0,57,0.000
4,docs\conf.py,4ec7d2a0d8eac4f915dc0d38a886cd57045bb0c4,2010-04-06,Armin Ronacher,Started working on documentation.\n\nSo far ju...,0,0,0,0,0.0,19,0,0,0.000,0,1062,0.000


In [12]:
new_df["pred_proba"] = model.predict_proba(new_df[FEATURE_COLS])[:, 1]
new_df["pred_label"] = (new_df["pred_proba"] >= 0.5).astype(int)

result_df = new_df.sort_values("pred_proba", ascending=False).reset_index(drop=True)
result_df.to_csv(RESULT_FILE, index=False)

result_df[["file", "pred_proba", "is_vuln", "pred_label", "date", "message"]].head(20)


,file,pred_proba,is_vuln,pred_label,date,message
0,tests\flask_tests.py,0.542693,0,1,2010-04-20,"Make it easier to associate a rule, endpoint a..."
1,tests\flask_tests.py,0.496863,0,0,2010-04-19,Documented some routing converter possibilities.
2,docs\conf.py,0.494144,0,0,2017-02-11,Migrate various docs links to https (#2180)\n\...
3,docs\conf.py,0.493260,0,0,2017-01-13,Use SOURCE_DATE_EPOCH for copyright year to ma...
4,docs\conf.py,0.489590,0,0,2017-05-04,optionally enable sphinxcontrib.log_cabinet\nc...
5,docs\conf.py,0.482330,0,0,2017-03-06,Fix typos/grammar in docs (#2201)
6,docs\conf.py,0.481732,0,0,2017-05-25,use alabaster
7,docs\conf.py,0.481724,0,0,2016-11-03,switch to packaged sphinx theme
8,docs\conf.py,0.469573,0,0,2017-08-01,fix links
9,docs\conf.py,0.456373,0,0,2016-06-26,clean up code formatting in some docs\nfix war...


In [13]:
def top_k_stats(df, ks=(10, 20, 50, 100)):
    rows = []
    total_real_vuln = int(df["is_vuln"].sum())

    for k in ks:
        part = df.head(k)
        found = int(part["is_vuln"].sum())
        precision_at_k = round(found / k, 4)
        recall_at_k = round(found / total_real_vuln, 4) if total_real_vuln else 0.0

        rows.append({
            "top_k": k,
            "real_vuln_in_top_k": found,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k
        })

    return pd.DataFrame(rows)

stats_df = top_k_stats(result_df)
stats_df


,top_k,real_vuln_in_top_k,precision_at_k,recall_at_k
0,10,0,0.00,0.0000
1,20,0,0.00,0.0000
2,50,0,0.00,0.0000
3,100,1,0.01,0.0625


In [14]:
real_vuln_positions = result_df[result_df["is_vuln"] == 1].copy()
real_vuln_positions["rank"] = range(1, len(real_vuln_positions) + 1)

# правильный реальный ранг в итоговой таблице
real_vuln_positions = result_df[result_df["is_vuln"] == 1].copy()
real_vuln_positions["rank"] = real_vuln_positions.index + 1

print("Total real vulnerable rows in new project:", len(real_vuln_positions))
real_vuln_positions[["rank", "file", "pred_proba", "date", "message"]].head(20)


Total real vulnerable rows in new project: 16


,rank,file,pred_proba,date,message
95,96,docs\conf.py,0.327376,2024-04-07,update project files (#5457)\n\n* update pre-c...
667,668,setup.py,0.240624,2019-04-15,Bump Jinja2 requirements\n\nThe 2.10.1 version...
787,788,tests\flask_tests.py,0.229396,2010-07-04,fixed possible security problem in module branch
981,982,tests\flask_tests.py,0.216572,2010-04-19,Fixed a security problem caused by changed sim...
999,1000,setup.py,0.214936,2016-04-03,Change to Pallets project\n\n* Update CHANGES\...
1075,1076,src\flask\json\tag.py,0.209120,2024-04-07,update project files (#5457)\n\n* update pre-c...
1232,1233,docs\conf.py,0.200216,2016-05-29,Convert readthedocs link for their .org -> .io...
1434,1435,src\flask\config.py,0.190727,2024-04-07,update project files (#5457)\n\n* update pre-c...
1634,1635,tests\test_json.py,0.180606,2024-04-07,update project files (#5457)\n\n* update pre-c...
1853,1854,flask.py,0.170677,2010-06-10,Added security document


In [15]:
top50_df = result_df.head(50).copy()
top50_df.to_csv(TOP50_FILE, index=False)

print("Top-50 risky files:")
top50_df[["file", "pred_proba", "is_vuln", "date", "message"]]


Top-50 risky files:


,file,pred_proba,is_vuln,date,message
0,tests\flask_tests.py,0.542693,0,2010-04-20,"Make it easier to associate a rule, endpoint a..."
1,tests\flask_tests.py,0.496863,0,2010-04-19,Documented some routing converter possibilities.
2,docs\conf.py,0.494144,0,2017-02-11,Migrate various docs links to https (#2180)\n\...
3,docs\conf.py,0.493260,0,2017-01-13,Use SOURCE_DATE_EPOCH for copyright year to ma...
4,docs\conf.py,0.489590,0,2017-05-04,optionally enable sphinxcontrib.log_cabinet\nc...
5,docs\conf.py,0.482330,0,2017-03-06,Fix typos/grammar in docs (#2201)
6,docs\conf.py,0.481732,0,2017-05-25,use alabaster
7,docs\conf.py,0.481724,0,2016-11-03,switch to packaged sphinx theme
8,docs\conf.py,0.469573,0,2017-08-01,fix links
9,docs\conf.py,0.456373,0,2016-06-26,clean up code formatting in some docs\nfix war...


In [16]:
summary = {
    "train_rows": int(len(train_df)),
    "train_vuln_rows": int(train_df["is_vuln"].sum()),
    "new_rows": int(len(result_df)),
    "new_real_vuln_rows": int(result_df["is_vuln"].sum()),
    "top10_real_vuln": int(result_df.head(10)["is_vuln"].sum()),
    "top20_real_vuln": int(result_df.head(20)["is_vuln"].sum()),
    "top50_real_vuln": int(result_df.head(50)["is_vuln"].sum()),
    "top100_real_vuln": int(result_df.head(100)["is_vuln"].sum())
}

pd.DataFrame([summary])


,train_rows,train_vuln_rows,new_rows,new_real_vuln_rows,top10_real_vuln,top20_real_vuln,top50_real_vuln,top100_real_vuln
0,5000,500,4241,16,0,0,0,1


In [17]:
try:
    from google.colab import files
    files.download(RESULT_FILE)
    files.download(TOP50_FILE)
except Exception:
    print("If you are not in Colab, result files are already saved locally.")


If you are not in Colab, result files are already saved locally.
